[Reference](https://generativeai.pub/rag-from-scratch-overview-pipeline-940a45c30e8f)

# Setting up the Environment

```
GOOGLE_API_KEY="your-google-api-key"
# LANGCHAIN_API_KEY="your-langchain-api-key"  # optional
# LANGCHAIN_TRACING_V2=True                   # optional
# LANGCHAIN_PROJECT="rag-from-scratch"       # optional
```

In [1]:
from doteanv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True) # mention the .env path

# Loading Documents

In [2]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

print(f"Total documents loaded: {len(docs)}")
# Total documents loaded: 1

Total documents loaded: 1


In [6]:
# Inspect the first document
print("Page content preview: \n", docs[0].page_content[:500])  # first 500 characters

print("\nMetadata:\n", docs[0].metadata)

Page content preview: 
 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In

Metadata:
 {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}


# Split Documents Into Chunks

In [8]:
!pip install langchain-text-splitters

In [11]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split Documents Into Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(f"Total chunks created: {len(splits)}")
# Total chunks created: 63

# Create Embeddings and Store Them


In [12]:
!pip install langchain_google_genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.3/713.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstalling google-genai-1.55.0:
      Successfully uninstalled google-genai-1.55.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, bu

In [12]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Embedding Model
embed_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# Create Embeddings and Store Them
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embed_model)

print("Vector store type:", type(vectorstore))

In [13]:
print("Number of vectors stored:", vectorstore._collection.count())
# Output: Number of vectors stored: 63

# Create a Retriever


In [14]:
# Create a Retriever
retriever = vectorstore.as_retriever()

# Validate Retrieval Independently
query = "What is Task Decomposition?"

retrieved_docs = retriever.invoke(query)

print(f"Number of retrieved documents: {len(retrieved_docs)}")

print("Top retrieved chunk:", retrieved_docs[0].page_content[:500])

# Format Retrieved Context


In [15]:
from langchain import hub

# Format Retrieved Context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt
prompt = hub.pull("rlm/rag-prompt")
"""
ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'},
messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={},
template="You are an assistant for question-answering tasks. Use the following
pieces of retrieved context to answer the question. If you don't know the
answer, just say that you don't know. Use three sentences maximum and keep the
answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])
"""

# Build the End-to-End RAG Chain

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp", temperature=0)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Execute the Pipeline

In [18]:
# Question
rag_chain.invoke("What is Task Decomposition?")